
# Erasmus+ Mobility Data Pipeline (2014–2023)

### Database Design, Data Processing and Performance Evaluation

**Course:** Computational Infrastructures for Data Science (CIDS)  
**Project:** Individual Project  
**Data source:** Erasmus+ geolocated mobility dataset (2014–2023)  
**Technology stack:** Python, pandas, PostgreSQL, Docker, GitHub

---

## 1. Introduction

### 1.1 Background

The Erasmus+ programme supports international educational mobility across Europe and participating partner countries.

The Erasmus+ mobility dataset provides an opportunity to investigate how large collections of regional mobility records can be stored, processed, and analyzed within a reproducible data science infrastructure.

### 1.2 Project Objective

The objective of this project is to design, implement, and evaluate a reproducible data processing pipeline for Erasmus+ mobility data between 2014 and 2023.

The project focuses on:

- Data collection, validation, and preprocessing.
- Structured data storage using PostgreSQL.
- SQL and Python-based mobility data processing.
- Containerization using Docker Compose.
- Performance evaluation in terms of execution time, memory consumption, and disk usage.
- Scalability assessment across different dataset sizes.

### 1.3 Research Questions

1. What are the main patterns of international Erasmus+ mobility between 2014 and 2023?
2. How efficiently can Erasmus+ mobility data be stored and processed using a relational database?
3. How does increasing data volume affect the computational performance of the pipeline?

### 1.4 Project Workflow

1. Data collection and inspection.
2. Data quality assessment and preprocessing.
3. Database design and data loading.
4. SQL and Python-based data aggregation.
5. Mobility analysis and result generation.
6. Containerization and reproducibility.
7. Performance benchmarking and scalability evaluation.



## 2. Dataset Description

### 2.1 Data Source

The project uses the `Erasmus_2014-2023_aggregate_NUTS.csv` dataset, containing aggregated Erasmus+ mobility flows between geographical regions.

The dataset covers the period from 2014 to 2023 and contains 588,109 records.

The original data is provided in CSV format with a semicolon separator.

### 2.2 Dataset Structure

| Variable | Description | Proposed SQL Data Type |
|---|---|---|
| `OD_ID` | Origin–destination pair identifier | VARCHAR |
| `ORIGIN` | Origin region code | VARCHAR |
| `DESTINATION` | Destination region code | VARCHAR |
| `YEAR` | Year of recorded mobility | INTEGER |
| `COUNT` | Number of mobility observations | BIGINT |

The unit of observation is an aggregated regional mobility flow for one year, rather than an individual student record.

### 2.3 Data Storage

The structured nature of the dataset makes it suitable for storage in a relational database such as PostgreSQL.

A relational database allows:

- Efficient filtering and aggregation of mobility records.
- Structured storage with defined data types.
- Indexing of frequently queried columns.
- SQL-based processing of mobility flows.
- Measurement of query performance and storage requirements.

### 2.4 Data Limitations

The dataset contains aggregated mobility flows rather than individual participant records. Therefore, the analysis focuses on geographical and temporal mobility patterns.


In [24]:
from pathlib import Path
import pandas as pd
import time

# Locate the project folder
ROOT = Path.cwd().resolve()

while not (ROOT / "pyproject.toml").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent
    
DATA_PATH = ROOT / "data/raw/Erasmus_2014-2023_aggregate_NUTS.csv"

# Measure data loading time
start = time.perf_counter()

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8-sig"
)

loading_time = time.perf_counter() - start

# Measure resource usage
file_size_mb = DATA_PATH.stat().st_size / 1024**2
dataframe_memory_mb = df.memory_usage(deep=True).sum() / 1024**2

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print(f"CSV loading time: {loading_time:.3f} seconds")
print(f"CSV file size: {file_size_mb:.2f} MiB")
print(f"DataFrame memory usage: {dataframe_memory_mb:.2f} MiB")

display(df.head())

Dataset loaded successfully!
Shape: (588109, 5)
CSV loading time: 0.159 seconds
CSV file size: 17.44 MiB
DataFrame memory usage: 103.20 MiB


,OD_ID,ORIGIN,DESTINATION,YEAR,COUNT
0,AL012_BG331,AL012,BG331,2020,4
1,AL012_BG411,AL012,BG411,2020,3
2,AL012_BG411,AL012,BG411,2021,6
3,AL012_CZ080,AL012,CZ080,2021,2
4,AL012_CZ080,AL012,CZ080,2022,2



## 3. Data Quality Assessment

Before conducting the analysis, the dataset was examined for missing values, duplicated observations, invalid counts, and inconsistent identifiers.

These checks help establish whether the data can be reliably aggregated for descriptive analysis.


In [25]:

quality = pd.DataFrame({
    "Missing values": df.isna().sum(),
    "Unique values": df.nunique()
})

display(quality)

print("Exact duplicate rows:", df.duplicated().sum())
print(
    "Duplicate origin-destination-year combinations:",
    df.duplicated(
        subset=["ORIGIN", "DESTINATION", "YEAR"]
    ).sum()
)
print("Negative counts:", (df["COUNT"] < 0).sum())
print("Zero counts:", (df["COUNT"] == 0).sum())

expected_id = (
    df["ORIGIN"] + "_" + df["DESTINATION"]
)

print(
    "Inconsistent OD_ID values:",
    (df["OD_ID"] != expected_id).sum()
)

print("\nColumn data types:")
display(df.dtypes)

print("Year range:", df["YEAR"].min(), "-", df["YEAR"].max())

print(
    "Invalid year values:",
    (~df["YEAR"].between(2014, 2023)).sum()
)

print(
    "Invalid region identifiers:",
    (
        (df["ORIGIN"].str.len() < 2) |
        (df["DESTINATION"].str.len() < 2)
    ).sum()
)

,Missing values,Unique values
OD_ID,0,183888
ORIGIN,0,1436
DESTINATION,0,1727
YEAR,0,10
COUNT,0,272


Exact duplicate rows: 0
Duplicate origin-destination-year combinations: 0
Negative counts: 0
Zero counts: 0
Inconsistent OD_ID values: 0

Column data types:


OD_ID            str
ORIGIN           str
DESTINATION      str
YEAR           int64
COUNT          int64
dtype: object

Year range: 2014 - 2023
Invalid year values: 0
Invalid region identifiers: 0



### 3.1 Findings

The dataset contains 588,109 aggregated observations and five variables.

The initial quality assessment identified:

- No missing values in the five variables.
- No exact duplicate rows.
- No repeated origin–destination–year combinations.
- No zero or negative mobility counts.
- No inconsistencies between the `OD_ID` variable and its origin–destination codes.

**Conclusion:** The dataset passed the initial structural quality checks and can be used for descriptive analysis. However, structural consistency does not independently establish completeness or comparability across years.



## 4. Data Preparation and Cleaning

### 4.1 Overview

Following the initial data quality assessment, the dataset was prepared for country-level and international mobility analysis.

The data preparation process involved:

1. Extracting country codes from regional identifiers.
2. Identifying domestic and international mobility.
3. Filtering international mobility observations.
4. Aggregating mobility counts by year.
5. Calculating country-level incoming, outgoing, and net mobility.

The original dataset was preserved throughout the process. All transformations were performed on copies of the original data.



### 4.2 Country Code Extraction

The original dataset contains regional identifiers for the origin and destination of mobility flows.

To enable country-level analysis, the first two characters of each regional code were extracted.

Two new variables were created:

- `ORIGIN_COUNTRY`: Country-code prefix of the origin region.
- `DEST_COUNTRY`: Country-code prefix of the destination region.

This allows regional mobility observations to be aggregated by country.


In [26]:
# Create a working copy
mobility = df.copy()

# Extract country codes
mobility["ORIGIN_COUNTRY"] = mobility["ORIGIN"].str[:2]
mobility["DEST_COUNTRY"] = mobility["DESTINATION"].str[:2]

print("Unique origin countries:", mobility["ORIGIN_COUNTRY"].nunique())
print("Unique destination countries:", mobility["DEST_COUNTRY"].nunique())

Unique origin countries: 37
Unique destination countries: 37



### 4.3 International Mobility Filtering

The project focuses on international Erasmus+ mobility between countries.

To distinguish international from domestic movements, the origin and destination country codes were compared.

- **Domestic mobility:** Origin and destination belong to the same country.
- **International mobility:** Origin and destination belong to different countries.

Domestic observations were excluded from subsequent international mobility analysis, while the original dataset was preserved.


In [27]:

# Identify domestic movements
mobility["SAME_COUNTRY"] = (
    mobility["ORIGIN_COUNTRY"] ==
    mobility["DEST_COUNTRY"]
)

# Separate domestic and international observations
domestic = mobility[
    mobility["SAME_COUNTRY"]
].copy()

international = mobility[
    ~mobility["SAME_COUNTRY"]
].copy()

# Create a summary table
summary = pd.DataFrame({
    "Category": ["Domestic", "International", "Total"],
    "Rows": [
        len(domestic),
        len(international),
        len(mobility)
    ],
    "Mobility Count": [
        domestic["COUNT"].sum(),
        international["COUNT"].sum(),
        mobility["COUNT"].sum()
    ]
})

summary["Share (%)"] = (
    summary["Mobility Count"] /
    mobility["COUNT"].sum() * 100
).round(2)

display(summary)


,Category,Rows,Mobility Count,Share (%)
0,Domestic,884,2556,0.1
1,International,587225,2551560,99.9
2,Total,588109,2554116,100.0



**Result and Interpretation**

The analysis identified 884 domestic observations, corresponding to 2,556 recorded movements (0.10% of the total).

International mobility accounts for 587,225 observations and 2,551,560 recorded movements (99.90% of the total).

Since the objective of this research is to investigate cross-border student mobility, only international observations will be used in the subsequent analysis.

The original dataset remains unchanged.



### 4.4 Annual Mobility Aggregation

To analyze changes in international Erasmus+ mobility over time, the dataset was aggregated by year.

Three indicators were calculated:

- **Mobility count:** Total number of recorded international movements per year.
- **Route count:** Number of recorded regional origin–destination combinations per year.
- **Annual growth rate:** Percentage change in mobility compared with the previous year.

The annual growth rate was calculated as:

$$
\text{Growth Rate}_t =
\frac{M_t-M_{t-1}}{M_{t-1}}\times100
$$

where $M_t$ represents the mobility count in year $t$.

This aggregation enables the identification of long-term trends and periods of substantial change.


In [28]:

# Aggregate international mobility by year
annual_mobility = (
    international.groupby("YEAR", as_index=False)
    .agg(
        mobility_count=("COUNT", "sum"),
        route_count=("COUNT", "size")
    )
    .sort_values("YEAR")
)

# Calculate annual percentage change
annual_mobility["growth_pct"] = (
    annual_mobility["mobility_count"]
    .pct_change() * 100
).round(2)

# Display results
display(annual_mobility)


,YEAR,mobility_count,route_count,growth_pct
0,2014,164107,45530,NaN
1,2015,271459,63583,65.42
2,2016,281644,65469,3.75
3,2017,289627,66122,2.83
4,2018,299374,67623,3.37
5,2019,299401,66308,0.01
6,2020,156916,44904,-47.59
7,2021,244404,54109,55.75
8,2022,266857,56728,9.19
9,2023,277771,56849,4.09



**Results and Interpretation**

Annual aggregation identifies a significant decline in recorded international mobility from 299,401 movements in 2019 to 156,916 in 2020 (-47.59%).

By 2023, mobility had recovered to 277,771 movements, approximately 92.8% of the 2019 level.

The aggregation provides a representative processing task for the subsequent database implementation and performance evaluation.

The results are descriptive. Changes in dataset coverage may affect comparisons across years.



### 4.5 Country-Level Mobility Balance

To examine differences between countries as senders and receivers of Erasmus+ mobility, three indicators were calculated:

- **Outgoing mobility:** Total recorded movements originating from a country.
- **Incoming mobility:** Total recorded movements arriving in a country.
- **Net mobility:** The difference between incoming and outgoing movements.

Net mobility was calculated as:

$$
\text{Net Mobility} = \text{Incoming Mobility} - \text{Outgoing Mobility}
$$

A positive net mobility balance indicates that a country receives more recorded movements than it sends, while a negative balance indicates the opposite.

The calculations were performed using international mobility observations only.


In [29]:

# Calculate outgoing mobility for each country and year
outgoing = (
    international.groupby(
        ["YEAR", "ORIGIN_COUNTRY"]
    )["COUNT"]
    .sum()
    .rename("outgoing")
    .rename_axis(["YEAR", "COUNTRY"])
)

# Calculate incoming mobility for each country and year
incoming = (
    international.groupby(
        ["YEAR", "DEST_COUNTRY"]
    )["COUNT"]
    .sum()
    .rename("incoming")
    .rename_axis(["YEAR", "COUNTRY"])
)

# Combine incoming and outgoing mobility
country_balance = pd.concat(
    [incoming, outgoing], axis=1
).fillna(0).reset_index()

country_balance[["incoming", "outgoing"]] = (
    country_balance[["incoming", "outgoing"]].astype(int)
)

# Calculate net mobility
country_balance["net_mobility"] = (
    country_balance["incoming"] -
    country_balance["outgoing"]
)

# Aggregate across 2014–2023
overall_balance = (
    country_balance.groupby("COUNTRY", as_index=False)
    [["incoming", "outgoing", "net_mobility"]]
    .sum()
)

print("Total countries:", len(overall_balance))

print("\nTop 10 net receiving countries:")
display(
    overall_balance.nlargest(
        10, "net_mobility"
    ).reset_index(drop=True)
)

print("\nTop 10 net sending countries:")
display(
    overall_balance.nsmallest(
        10, "net_mobility"
    ).reset_index(drop=True)
)

# Validate results
assert country_balance["incoming"].sum() == international["COUNT"].sum()
assert country_balance["outgoing"].sum() == international["COUNT"].sum()
assert country_balance["net_mobility"].sum() == 0

print("All consistency checks passed.")


Total countries: 37

Top 10 net receiving countries:


,COUNTRY,incoming,outgoing,net_mobility
0,UK,183911,110445,73466
1,SE,87388,37102,50286
2,PT,123622,77095,46527
3,ES,397720,353395,44325
4,NO,61706,23063,38643
5,IE,63817,31873,31944
6,FI,69050,41375,27675
7,PL,131078,107078,24000
8,BE,96560,72742,23818
9,MT,19371,2761,16610



Top 10 net sending countries:


,COUNTRY,incoming,outgoing,net_mobility
0,FR,236989,366996,-130007
1,IT,228882,325020,-96138
2,DE,252551,347870,-95319
3,TR,38857,118900,-80043
4,RO,27576,44177,-16601
5,NL,110625,124699,-14074
6,SK,16470,25066,-8596
7,EL,36779,42579,-5800
8,BG,11254,16565,-5311
9,AL,544,3527,-2983


All consistency checks passed.



**Results and Interpretation**

The country-level aggregation identifies differences between sending and receiving countries.

The United Kingdom recorded the largest positive net mobility balance (+73,466), while France recorded the largest negative balance (-130,007).

All consistency checks passed:

- Total incoming mobility equals total international mobility.
- Total outgoing mobility equals total international mobility.
- The sum of net mobility balances equals zero.

These checks confirm the internal consistency of the aggregation.

The country-level mobility calculation will be used as a representative analytical workload in the performance evaluation.

Absolute mobility balances should not be interpreted as direct measures of educational attractiveness.



## 5. Mobility Data Analysis

This section demonstrates the analytical capabilities of the Erasmus+ data processing pipeline.

The analysis focuses on three representative tasks:

1. Country-level mobility rankings and balances.
2. Normalized mobility imbalance.
3. Bilateral mobility flows.

These tasks demonstrate data aggregation, filtering, sorting, and transformation, and provide analytical workloads for subsequent database performance evaluation.

### 5.1 Country-Level Mobility Rankings


In [30]:

country_names = {
    "AL": "Albania",
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "CH": "Switzerland",
    "CY": "Cyprus",
    "CZ": "Czechia",
    "DE": "Germany",
    "DK": "Denmark",
    "EE": "Estonia",
    "EL": "Greece",
    "ES": "Spain",
    "FI": "Finland",
    "FR": "France",
    "HR": "Croatia",
    "HU": "Hungary",
    "IE": "Ireland",
    "IS": "Iceland",
    "IT": "Italy",
    "LI": "Liechtenstein",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "ME": "Montenegro",
    "MK": "North Macedonia",
    "MT": "Malta",
    "NL": "Netherlands",
    "NO": "Norway",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "RS": "Serbia",
    "SE": "Sweden",
    "SI": "Slovenia",
    "SK": "Slovakia",
    "TR": "Türkiye",
    "UK": "United Kingdom"
}

# Add full country names
overall_balance["country_name"] = (
    overall_balance["COUNTRY"].map(country_names)
)

# Check that all codes were mapped
assert overall_balance["country_name"].notna().all()

display(overall_balance.head())


,COUNTRY,incoming,outgoing,net_mobility,country_name
0,AL,544,3527,-2983,Albania
1,AT,65034,55549,9485,Austria
2,BE,96560,72742,23818,Belgium
3,BG,11254,16565,-5311,Bulgaria
4,CH,2011,1,2010,Switzerland


In [31]:

# Top 10 sending countries
top_senders = overall_balance.nlargest(
    10, "outgoing"
)

# Top 10 receiving countries
top_receivers = overall_balance.nlargest(
    10, "incoming"
)

print("Top 10 sending countries:")
display(
    top_senders[
        ["country_name", "outgoing"]
    ].reset_index(drop=True)
)

print("Top 10 receiving countries:")
display(
    top_receivers[
        ["country_name", "incoming"]
    ].reset_index(drop=True)
)


Top 10 sending countries:


,country_name,outgoing
0,France,366996
1,Spain,353395
2,Germany,347870
3,Italy,325020
4,Netherlands,124699
5,Türkiye,118900
6,United Kingdom,110445
7,Poland,107078
8,Portugal,77095
9,Belgium,72742


Top 10 receiving countries:


,country_name,incoming
0,Spain,397720
1,Germany,252551
2,France,236989
3,Italy,228882
4,United Kingdom,183911
5,Poland,131078
6,Portugal,123622
7,Netherlands,110625
8,Belgium,96560
9,Sweden,87388



**Results**

France recorded the highest outgoing mobility count, while Spain was the leading receiving country with 397,720 recorded international movements.

The rankings demonstrate the pipeline's ability to aggregate mobility flows by country and retrieve the largest values.

Absolute mobility counts should not be interpreted as direct measures of educational attractiveness.



### 5.2 Country Mobility Balances

Net mobility measures the difference between incoming and outgoing mobility:

$$
B_i = I_i - O_i
$$

where $I_i$ and $O_i$ represent incoming and outgoing mobility for country $i$.

Positive values identify net receiving countries, while negative values identify net sending countries.


In [32]:

# Sort countries by net mobility
balance_sorted = overall_balance.sort_values(
    "net_mobility",
    ascending=False
)

# Identify net receivers, senders and balanced countries
net_receivers = balance_sorted[
    balance_sorted["net_mobility"] > 0
]

net_senders = balance_sorted[
    balance_sorted["net_mobility"] < 0
]

balanced = balance_sorted[
    balance_sorted["net_mobility"] == 0
]

print("Total countries:", len(balance_sorted))
print("Net receiving countries:", len(net_receivers))
print("Net sending countries:", len(net_senders))
print("Balanced countries:", len(balanced))

display(
    balance_sorted[
        ["country_name", "incoming",
         "outgoing", "net_mobility"]
    ].reset_index(drop=True)
)


Total countries: 37
Net receiving countries: 22
Net sending countries: 15
Balanced countries: 0


,country_name,incoming,outgoing,net_mobility
0,United Kingdom,183911,110445,73466
1,Sweden,87388,37102,50286
2,Portugal,123622,77095,46527
3,Spain,397720,353395,44325
4,Norway,61706,23063,38643
5,Ireland,63817,31873,31944
6,Finland,69050,41375,27675
7,Poland,131078,107078,24000
8,Belgium,96560,72742,23818
9,Malta,19371,2761,16610


In [33]:
# Validate the consistency of country-level balances
total_surplus = net_receivers["net_mobility"].sum()

total_deficit = abs(
    net_senders["net_mobility"].sum()
)

assert total_surplus == total_deficit

print(f"Total positive balance: {total_surplus:,}")
print(f"Total negative balance: {total_deficit:,}")
print("Mobility balance validation passed.")

Total positive balance: 460,815
Total negative balance: 460,815
Mobility balance validation passed.



**Results and Validation**

The United Kingdom recorded the largest positive mobility balance (+73,466), while France recorded the largest negative balance (-130,007).

The total positive balance equals the absolute total negative balance.

This consistency check confirms that international mobility flows are correctly represented in the country-level aggregation.



### 5.3 Relative Mobility Imbalance

The Normalized Mobility Balance Index (NMBI) measures mobility imbalance relative to total mobility volume.

$$
NMBI_i = \frac{I_i-O_i}{I_i+O_i}
$$

The index ranges from -1 to +1:

- Negative values indicate a net sending orientation.
- Positive values indicate a net receiving orientation.
- Zero indicates balanced mobility.

This calculation demonstrates the generation of derived indicators from aggregated mobility data.


In [34]:

# Total mobility associated with each country
overall_balance["total_mobility"] = (
    overall_balance["incoming"] +
    overall_balance["outgoing"]
)

# Calculate normalized mobility balance
overall_balance["NMBI"] = (
    overall_balance["net_mobility"] /
    overall_balance["total_mobility"].replace(
        0, float("nan")
    )
)

# Rank countries by normalized balance
relative_balance = overall_balance.sort_values(
    "NMBI",
    ascending=False
)

print("Top 10 relative net receivers:")

display(
    relative_balance.head(10)[
        ["country_name", "incoming", "outgoing",
         "net_mobility", "NMBI"]
    ].round(3)
)

print("Top 10 relative net senders:")

display(
    relative_balance.tail(10)
    .sort_values("NMBI")[
        ["country_name", "incoming", "outgoing",
         "net_mobility", "NMBI"]
    ].round(3)
)

# Validate the index
assert overall_balance["NMBI"].between(-1, 1).all()


Top 10 relative net receivers:


,country_name,incoming,outgoing,net_mobility,NMBI
4,Switzerland,2011,1,2010,0.999
25,Malta,19371,2761,16610,0.750
17,Iceland,7244,2561,4683,0.478
27,Norway,61706,23063,38643,0.456
32,Sweden,87388,37102,50286,0.404
21,Luxembourg,10107,4612,5495,0.373
5,Cyprus,9830,4756,5074,0.348
16,Ireland,63817,31873,31944,0.334
19,Liechtenstein,671,348,323,0.317
9,Estonia,14328,8192,6136,0.272


Top 10 relative net senders:


,country_name,incoming,outgoing,net_mobility,NMBI
0,Albania,544,3527,-2983,-0.733
23,Montenegro,283,1125,-842,-0.598
35,Türkiye,38857,118900,-80043,-0.507
31,Serbia,1994,4453,-2459,-0.381
24,North Macedonia,1021,1853,-832,-0.289
30,Romania,27576,44177,-16601,-0.231
13,France,236989,366996,-130007,-0.215
34,Slovakia,16470,25066,-8596,-0.207
3,Bulgaria,11254,16565,-5311,-0.191
18,Italy,228882,325020,-96138,-0.174



**Results and Validation**

The normalized mobility balance provides a complementary perspective to absolute mobility counts.

Spain, despite being the largest receiving country, recorded an NMBI of approximately +0.059, indicating relatively balanced mobility.

All calculated NMBI values fall within the expected range of -1 to +1.

The indicator does not account for differences in population, programme participation, or dataset coverage. Switzerland, in particular, has limited comparability due to its separate mobility programme.

Source: [Movetia — Swiss Programme for international exchanges](https://movetia.ch/en/how-to/guidance-and-funding-practice/swiss-programme-for-erasmus).



### 5.4 Bilateral Mobility Connections

Bilateral mobility analysis examines international mobility flows between pairs of countries.

The analysis includes:

- Directional mobility flows between origin and destination countries.
- A country-to-country mobility matrix.
- Combined bilateral mobility volumes.
- Bilateral reciprocity indicators.

This section demonstrates multi-column aggregation, data transformation, and matrix construction.

The directional mobility aggregation provides a representative workload for SQL query performance evaluation.


In [35]:

# Aggregate mobility by country pair
bilateral_flows = (
    international.groupby(
        ["ORIGIN_COUNTRY", "DEST_COUNTRY"],
        as_index=False
    )["COUNT"]
    .sum()
)

# Add country names
bilateral_flows["origin_name"] = (
    bilateral_flows["ORIGIN_COUNTRY"]
    .map(country_names)
)

bilateral_flows["destination_name"] = (
    bilateral_flows["DEST_COUNTRY"]
    .map(country_names)
)

# Rank directional mobility flows
top_flows = bilateral_flows.sort_values(
    "COUNT", ascending=False
)

print("Number of observed directional connections:",
      len(bilateral_flows))

display(
    top_flows.head(15)[
        ["origin_name", "destination_name", "COUNT"]
    ].reset_index(drop=True)
)

# Validate the aggregation
assert (
    bilateral_flows["COUNT"].sum()
    == international["COUNT"].sum()
)


Number of observed directional connections: 1217


,origin_name,destination_name,COUNT
0,Italy,Spain,99230
1,Spain,Italy,80738
2,France,Spain,64966
3,Germany,Spain,58997
4,Germany,France,47606
5,France,United Kingdom,43283
6,France,Germany,41770
7,Italy,France,41013
8,Italy,Germany,35388
9,Germany,United Kingdom,33621


In [36]:

# Create a country-to-country mobility matrix
mobility_matrix = bilateral_flows.pivot(
    index="ORIGIN_COUNTRY",
    columns="DEST_COUNTRY",
    values="COUNT"
).fillna(0)

# Ensure the same countries appear on both axes
all_countries = sorted(overall_balance["COUNTRY"])

mobility_matrix = mobility_matrix.reindex(
    index=all_countries,
    columns=all_countries,
    fill_value=0
).astype(int)

print("Mobility matrix shape:", mobility_matrix.shape)

display(
    mobility_matrix.loc[
        ["IT", "ES", "FR", "DE", "UK"],
        ["IT", "ES", "FR", "DE", "UK"]
    ]
)

assert mobility_matrix.to_numpy().sum() == international["COUNT"].sum()


Mobility matrix shape: (37, 37)


DEST_COUNTRY,IT,ES,FR,DE,UK
ORIGIN_COUNTRY,,,,,
IT,0,99230,41013,35388,18705
ES,80738,0,32435,31769,27807
FR,28748,64966,0,41770,43283
DE,24362,58997,47606,0,33621
UK,7604,27295,27310,14934,0



#### Bilateral Mobility Volume and Reciprocity

The combined mobility volume between countries $i$ and $j$ is defined as:

$$
V_{ij} = F_{ij} + F_{ji}
$$

Bilateral reciprocity is calculated as:

$$
R_{ij} = \frac{2\min(F_{ij},F_{ji})}{F_{ij}+F_{ji}}
$$

The reciprocity index ranges from 0 (one-directional mobility) to 1 (perfectly balanced mobility).

Pairs with zero mobility in both directions are excluded.

This calculation demonstrates the transformation of directional flows into unique bilateral relationships.


In [37]:

from itertools import combinations

# Store bilateral results
pair_results = []

# Generate each unordered country pair once
for country_a, country_b in combinations(
    all_countries, 2
):

    # Mobility in both directions
    flow_ab = mobility_matrix.loc[
        country_a, country_b
    ]

    flow_ba = mobility_matrix.loc[
        country_b, country_a
    ]

    # Combined bilateral mobility
    total_volume = flow_ab + flow_ba

    # Exclude pairs without recorded mobility
    if total_volume == 0:
        continue

    # Reciprocity index
    reciprocity = (
        2 * min(flow_ab, flow_ba)
        / total_volume
    )

    pair_results.append({
        "country_a": country_names[country_a],
        "country_b": country_names[country_b],
        "flow_a_to_b": flow_ab,
        "flow_b_to_a": flow_ba,
        "bilateral_volume": total_volume,
        "reciprocity": reciprocity
    })

# Convert results into a DataFrame
bilateral_pairs = pd.DataFrame(pair_results)

# Sort by combined bilateral mobility
bilateral_pairs = bilateral_pairs.sort_values(
    "bilateral_volume", ascending=False
).reset_index(drop=True)

print("Active bilateral connections:",
      len(bilateral_pairs))

display(
    bilateral_pairs.head(15).round(3)
)

# Validate results
assert bilateral_pairs["reciprocity"].between(0, 1).all()

assert (
    bilateral_pairs["bilateral_volume"].sum()
    == international["COUNT"].sum()
)


Active bilateral connections: 639


,country_a,country_b,flow_a_to_b,flow_b_to_a,bilateral_volume,reciprocity
0,Spain,Italy,80738,99230,179968,0.897
1,Spain,France,32435,64966,97401,0.666
2,Germany,Spain,58997,31769,90766,0.700
3,Germany,France,47606,41770,89376,0.935
4,France,United Kingdom,43283,27310,70593,0.774
5,France,Italy,28748,41013,69761,0.824
6,Germany,Italy,24362,35388,59750,0.815
7,Spain,United Kingdom,27807,27295,55102,0.991
8,Spain,Poland,33062,19870,52932,0.751
9,Germany,United Kingdom,33621,14934,48555,0.615


In [38]:
# Maximum number of possible bilateral connections
possible_pairs = (
    len(all_countries) *
    (len(all_countries) - 1) // 2
)

# Number of observed bilateral connections
observed_pairs = len(bilateral_pairs)

# Connection density
connection_density = observed_pairs / possible_pairs

# Average bilateral reciprocity
mean_reciprocity = bilateral_pairs["reciprocity"].mean()

print("Possible bilateral connections:", possible_pairs)
print("Observed bilateral connections:", observed_pairs)

print(f"Connection density: {connection_density:.2%}")
print(f"Mean reciprocity: {mean_reciprocity:.3f}")

Possible bilateral connections: 666
Observed bilateral connections: 639
Connection density: 95.95%
Mean reciprocity: 0.537



**Results and Validation**

Among 666 possible bilateral relationships between 37 countries, 639 recorded mobility flows during 2014–2023.

The observed connection density was 95.95%, while the mean bilateral reciprocity was approximately 0.537.

The validation checks confirm that:

- All reciprocity values are within the expected range.
- The sum of bilateral mobility volumes equals total international mobility.

These results demonstrate the correctness of the bilateral data transformation.

The analysis aggregates mobility across the entire study period and does not capture annual changes in individual connections.



### 5.5 Processed Data Outputs

The pipeline generates structured analytical outputs representing annual, country-level, and bilateral Erasmus+ mobility.

The processed tables can be exported for inspection and validation.

These outputs will also serve as reference results for validating the PostgreSQL implementation.


In [39]:
# Create output directory
ANALYSIS_DIR = ROOT / "data/processed/analysis"

ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save country-level results
overall_balance.to_csv(
    ANALYSIS_DIR / "country_balance.csv",
    index=False
)

# Save directional mobility flows
bilateral_flows.to_csv(
    ANALYSIS_DIR / "bilateral_flows.csv",
    index=False
)

# Save bilateral mobility statistics
bilateral_pairs.to_csv(
    ANALYSIS_DIR / "bilateral_pairs.csv",
    index=False
)

# Save mobility matrix
mobility_matrix.to_csv(
    ANALYSIS_DIR / "mobility_matrix.csv"
)

print("Processed analytical tables saved successfully.")

for file in sorted(ANALYSIS_DIR.glob("*.csv")):
    print(
        f"{file.name}: "
        f"{file.stat().st_size / 1024:.2f} KiB"
    )

Processed analytical tables saved successfully.
annual_mobility.csv: 0.28 KiB
bilateral_flows.csv: 33.50 KiB
bilateral_pairs.csv: 29.50 KiB
country_balance.csv: 2.11 KiB
country_balance_by_year.csv: 8.43 KiB
country_balance_overall.csv: 2.11 KiB
international_country_flows.csv: 155.23 KiB
international_trends.csv: 0.22 KiB
mobility_matrix.csv: 5.20 KiB



### 5.5 Processed Data Outputs

The analytical results are exported as structured CSV tables.

These tables serve as reference outputs for validating the subsequent PostgreSQL implementation.

The exported files include annual mobility statistics, country-level balances, directional mobility flows, bilateral relationships, and the mobility matrix.


In [40]:
ANALYSIS_DIR = ROOT / "data/processed/analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

annual_mobility.to_csv(
    ANALYSIS_DIR / "annual_mobility.csv",
    index=False
)

overall_balance.to_csv(
    ANALYSIS_DIR / "country_balance_overall.csv",
    index=False
)

bilateral_flows.to_csv(
    ANALYSIS_DIR / "bilateral_flows.csv",
    index=False
)

bilateral_pairs.to_csv(
    ANALYSIS_DIR / "bilateral_pairs.csv",
    index=False
)

mobility_matrix.to_csv(
    ANALYSIS_DIR / "mobility_matrix.csv"
)

print("Reference analysis tables saved successfully.")

Reference analysis tables saved successfully.



## 6. Database Design and Implementation

### 6.1 Database Selection

PostgreSQL 16 was selected as the primary database management system because the Erasmus+ dataset has a structured relational format and requires filtering, aggregation, indexing, and integrity validation.

The database provides:

- Persistent and structured data storage.
- SQL-based analytical processing.
- Data integrity constraints.
- Indexing capabilities.
- Transaction management.
- Performance and storage measurement tools.

### 6.2 Database Schema

The database contains one main table, `mobility`.

| Column | Data Type | Description |
|---|---|---|
| id | BIGINT | Automatically generated primary key |
| od_id | TEXT | Origin-destination identifier |
| origin | TEXT | Origin region code |
| destination | TEXT | Destination region code |
| year | INTEGER | Mobility year |
| count | BIGINT | Mobility count |
| origin_country | VARCHAR(2) | Generated origin country code |
| dest_country | VARCHAR(2) | Generated destination country code |
| same_region | BOOLEAN | Generated regional mobility indicator |
| same_country | BOOLEAN | Generated domestic mobility indicator |

The first five dataset attributes are imported from the original CSV file. The primary key and four derived attributes are generated automatically by PostgreSQL.

### 6.3 Data Integrity

The database schema implements several integrity constraints:

- PRIMARY KEY: ensures unique record identification.
- NOT NULL: prevents missing values in source attributes.
- UNIQUE: prevents duplicate origin-destination-year combinations.
- CHECK: validates mobility counts, year ranges, geographical identifiers, and OD_ID consistency.

These constraints transfer part of the data validation process from Python to the database.

### 6.4 Data Loading

The original CSV dataset is loaded using Python and PostgreSQL COPY.

The loader reads the CSV sequentially instead of loading the entire dataset into memory. This approach reduces the Python memory requirements.

The loading process is transactional. Existing records are removed before reloading the dataset, preventing duplication.

The complete database contains 588,109 records.


In [41]:

import sys

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from database import get_connection


def query_dataframe(query):

    with get_connection() as conn:
        with conn.cursor() as cur:

            cur.execute(query)

            columns = [
                col.name for col in cur.description
            ]

            rows = cur.fetchall()

    return pd.DataFrame(rows, columns=columns)


database_validation = query_dataframe("""
    SELECT
        COUNT(*) AS total_records,
        SUM(count) AS total_mobility,

        COUNT(*) FILTER (
            WHERE same_country = FALSE
        ) AS international_records,

        SUM(count) FILTER (
            WHERE same_country = FALSE
        ) AS international_mobility,

        MIN(year) AS first_year,
        MAX(year) AS last_year

    FROM mobility;
""")

display(database_validation)

assert database_validation.loc[0, "total_records"] == 588109
assert database_validation.loc[0, "total_mobility"] == 2554116
assert database_validation.loc[0, "international_records"] == 587225
assert database_validation.loc[0, "international_mobility"] == 2551560

print("Database validation passed.")


,total_records,total_mobility,international_records,international_mobility,first_year,last_year
0,588109,2554116,587225,2551560,2014,2023


Database validation passed.



### 6.5 SQL-Based Analytical Processing

Four representative analytical workloads were implemented directly in PostgreSQL:

1. Annual international mobility aggregation.
2. Country-level incoming and outgoing mobility balance.
3. Normalized Mobility Balance Index (NMBI).
4. Bilateral directional mobility flows.

The SQL implementation uses GROUP BY, aggregate functions, common table expressions, FULL OUTER JOIN, COALESCE, and NULLIF.

The original pandas implementation serves as a reference for validating the database results.

The SQL validation confirmed:

| Validation Metric | Result |
|---|---:|
| Total records | 588,109 |
| Total mobility count | 2,554,116 |
| International records | 587,225 |
| International mobility count | 2,551,560 |
| Number of years | 10 |
| Directional connections | 1,217 |

The agreement between the pandas and SQL aggregate results confirms the consistency of the implemented data processing pipeline.



## 7. Docker Containerization

### 7.1 System Architecture

The project uses a containerized architecture consisting of two services:

- db: PostgreSQL 16 database.
- pipeline: Python 3.12 data processing application.

Docker Compose manages communication, service dependencies, environment variables, and persistent storage.

The Python application connects to PostgreSQL through the internal Docker network using the service name `db`.

### 7.2 Architecture Diagram

```text
Erasmus+ CSV Dataset
         |
         v
Python Pipeline Container
         |
         | PostgreSQL COPY
         v
PostgreSQL Database Container
         |
         | SQL analytical queries
         v
Python Pipeline Container
         |
         v
Processed SQL Tables (CSV)
```

### 7.3 Persistent Storage

A Docker named volume preserves PostgreSQL data between container restarts.

The original dataset is mounted as a read-only volume, while analytical results are written to a shared output directory.

Sensitive database configuration is stored in an excluded `.env` file.

### 7.4 Automated Pipeline

The pipeline performs four main stages:

1. Database initialization.
2. Dataset loading.
3. Database validation.
4. SQL analysis and result export.

The entire system can be started using:

```bash
docker compose up --build
```

The successful execution of the Python container returns exit code 0.

The PostgreSQL container remains available for subsequent queries and performance experiments.



## 8. Performance Evaluation

### 8.1 Experimental Environment

All performance experiments were conducted locally using Docker Desktop and WSL 2.

| Specification | Value |
|---|---|
| Processor | Intel Core i5-1240P |
| Physical cores | 12 |
| Logical processors | 16 |
| Host RAM | 15.63 GiB |
| Docker available RAM | 7.57 GiB |
| Operating system | Windows |
| Virtualization | WSL 2 |
| Python version | 3.12 |
| PostgreSQL version | 16.16 |
| Docker Engine | 29.9.3 |
| Docker Compose | 5.7.0 |

### 8.2 Experimental Design

Three dataset sizes were evaluated:

- 25%: 147,027 records.
- 50%: 294,054 records.
- 100%: 588,109 records.

Random sampling was performed using a fixed random seed of 42. The smaller samples are nested within the larger samples.

Each dataset size was evaluated three times, resulting in nine benchmark runs.

### 8.3 Measurement Methodology

Execution time was measured using Python's `time.perf_counter()`.

The measured pipeline includes:

- Database initialization.
- Data loading.
- PostgreSQL statistics refresh using ANALYZE.
- Data validation.
- SQL analytical processing and result export.

Sample preparation and Docker image construction are excluded from the measured pipeline time.

Database storage was measured using:

- pg_database_size()
- pg_table_size()
- pg_indexes_size()
- pg_total_relation_size()

Peak container memory was measured using the Linux cgroup memory.peak counter.

Memory measurements were conducted separately on the complete dataset.

### 8.4 Experimental Limitations

The experiments were conducted on one computer using a local Docker environment.

The operating system and database caches were not explicitly cleared between runs.

Therefore, the results describe performance under the tested conditions and should not be generalized to all hardware or significantly larger datasets.


In [42]:

BENCHMARK_DIR = ROOT / "docs/benchmarks"

benchmark_results = pd.read_csv(
    BENCHMARK_DIR / "benchmark_results.csv"
)

benchmark_summary = pd.read_csv(
    BENCHMARK_DIR / "benchmark_summary.csv"
)

assert len(benchmark_results) == 9

assert len(benchmark_summary) == 3

display(benchmark_summary)


,dataset_fraction,records,csv_bytes,loading_seconds,analyze_seconds,annual_seconds,balance_seconds,imbalance_seconds,bilateral_seconds,pipeline_seconds,database_bytes,table_bytes,index_bytes,total_relation_bytes
0,0.25,147027.0,4718858.0,0.656191,0.107239,0.017664,0.049455,0.050822,0.030735,0.949404,28654615.0,12787712.0,7970816.0,20758528.0
1,0.50,294054.0,9437692.0,1.197478,0.121686,0.027785,0.063134,0.061339,0.038780,1.551118,48684055.0,24846336.0,15933440.0,40779776.0
2,1.00,588109.0,18287184.0,2.837561,0.141755,0.040383,0.087440,0.088648,0.058757,3.303215,89742359.0,50012160.0,31817728.0,81829888.0


In [43]:

performance_table = pd.DataFrame({

    "Dataset": (
        benchmark_summary["dataset_fraction"] * 100
    ).astype(int).astype(str) + "%",

    "Records": benchmark_summary[
        "records"
    ].astype(int),

    "Loading (s)": benchmark_summary[
        "loading_seconds"
    ],

    "Annual query (ms)": benchmark_summary[
        "annual_seconds"
    ] * 1000,

    "Balance query (ms)": benchmark_summary[
        "balance_seconds"
    ] * 1000,

    "NMBI query (ms)": benchmark_summary[
        "imbalance_seconds"
    ] * 1000,

    "Bilateral query (ms)": benchmark_summary[
        "bilateral_seconds"
    ] * 1000,

    "Pipeline (s)": benchmark_summary[
        "pipeline_seconds"
    ],

    "CSV (MiB)": benchmark_summary[
        "csv_bytes"
    ] / 1024**2,

    "Database (MiB)": benchmark_summary[
        "database_bytes"
    ] / 1024**2,

    "Table + indexes (MiB)": benchmark_summary[
        "total_relation_bytes"
    ] / 1024**2,

})

display(
    performance_table
    .set_index("Dataset")
    .T.round(3)
)


Dataset,25%,50%,100%
Records,147027.000,294054.000,588109.000
Loading (s),0.656,1.197,2.838
Annual query (ms),17.664,27.785,40.383
Balance query (ms),49.455,63.134,87.440
NMBI query (ms),50.822,61.339,88.648
Bilateral query (ms),30.735,38.780,58.757
Pipeline (s),0.949,1.551,3.303
CSV (MiB),4.500,9.000,17.440
Database (MiB),27.327,46.429,85.585
Table + indexes (MiB),19.797,38.891,78.039


In [44]:

runtime_statistics = (
    benchmark_results
    .groupby("dataset_fraction")["pipeline_seconds"]
    .agg(["mean", "std", "min", "max"])
)

runtime_statistics.index = ["25%", "50%", "100%"]

display(runtime_statistics.round(4))


,mean,std,min,max
25%,0.9494,0.0409,0.9242,0.9966
50%,1.5511,0.0534,1.4959,1.6025
100%,3.3032,0.1676,3.1762,3.4931



### 8.5 Memory Consumption

Peak memory consumption was measured separately for the Python pipeline and PostgreSQL containers while processing the complete dataset.

| Container | Peak Memory (MiB) |
|---|---:|
| Python pipeline | 98.84 |
| PostgreSQL | 289.13 |
| Sum of individual peaks | 387.97 |

The Python pipeline maintained relatively low memory consumption because the source CSV was processed sequentially rather than loaded entirely into a pandas DataFrame.

PostgreSQL required additional memory for data loading, query execution, internal buffers, and filesystem caching.

The sum of the individual container peaks represents an upper bound, not a directly measured simultaneous system-wide peak.

The memory measurements were obtained from a single full-dataset execution using the Linux cgroup memory.peak counter.

The measured memory consumption remained substantially below the available Docker memory of 7.57 GiB.



### 8.6 Performance Interpretation

The complete dataset was processed in an average of approximately 3.303 seconds.

Data loading was the dominant computational stage, requiring approximately 2.838 seconds for the complete dataset.

All four analytical SQL queries completed in less than 100 milliseconds on average.

The country balance and relative imbalance queries required more processing time than the annual mobility query because they involve multiple aggregations and joins.

The original dataset occupies approximately 17.44 MiB, while the complete PostgreSQL database occupies approximately 85.58 MiB.

This additional storage is associated with generated attributes, indexes, internal PostgreSQL storage structures, and database metadata.

The results demonstrate a trade-off between storage requirements and database functionality.



## 9. Scalability Analysis

The scalability experiment evaluates how the pipeline responds to increasing data volume.

Three nested dataset sizes were processed under the same experimental configuration.

The objective is to investigate changes in:

- Data loading time.
- Analytical query execution time.
- Total pipeline execution time.
- Database storage requirements.


In [45]:

small = benchmark_summary.loc[
    benchmark_summary["dataset_fraction"] == 0.25
].iloc[0]

full = benchmark_summary.loc[
    benchmark_summary["dataset_fraction"] == 1.00
].iloc[0]

scalability = pd.DataFrame({

    "Metric": [
        "Number of records",
        "Loading time",
        "Annual query",
        "Country balance query",
        "Relative imbalance query",
        "Bilateral query",
        "Full pipeline time",
        "Table and index storage"
    ],

    "Growth factor": [

        full["records"] /
        small["records"],

        full["loading_seconds"] /
        small["loading_seconds"],

        full["annual_seconds"] /
        small["annual_seconds"],

        full["balance_seconds"] /
        small["balance_seconds"],

        full["imbalance_seconds"] /
        small["imbalance_seconds"],

        full["bilateral_seconds"] /
        small["bilateral_seconds"],

        full["pipeline_seconds"] /
        small["pipeline_seconds"],

        full["total_relation_bytes"] /
        small["total_relation_bytes"],

    ]

})

display(scalability.round(3))


,Metric,Growth factor
0,Number of records,4.000
1,Loading time,4.324
2,Annual query,2.286
3,Country balance query,1.768
4,Relative imbalance query,1.744
5,Bilateral query,1.912
6,Full pipeline time,3.479
7,Table and index storage,3.942



### 9.1 Scalability Results and Interpretation

Increasing the dataset size from 25% to 100% results in an approximately fourfold increase in the number of records.

Data loading time increases by approximately 4.32 times, suggesting approximately linear scaling over the tested range.

The total pipeline execution time increases by approximately 3.48 times.

Analytical query execution times increase more slowly than the number of records, with growth factors ranging from approximately 1.74 to 2.29.

The combined table and index storage increases by approximately 3.94 times, indicating approximately linear storage scaling.

The results suggest that the implemented architecture handles the tested dataset sizes efficiently.

However, the experiments cover only three dataset sizes. Therefore, the observed scaling behavior cannot be extrapolated reliably to substantially larger workloads.

Memory scalability was not evaluated separately because peak memory was measured only for the complete dataset.



### 9.2 Index Performance Evaluation

An additional experiment evaluates the effect of a composite partial index on a selective analytical query.

The query filters international mobility records by origin country and year.

The comparison considers:

- Query execution time without the additional index.
- Query execution time with the additional index.
- PostgreSQL execution plan and scan type.
- Index creation time.
- Additional storage requirements.

Each condition is evaluated three times using EXPLAIN ANALYZE.

The experimental index is removed after the benchmark to preserve the original database configuration.

Experimental results will be added after the index benchmark is completed.


In [46]:

index_results = pd.read_csv(
    BENCHMARK_DIR / "index_benchmark.csv"
)

index_summary = (
    index_results
    .groupby("condition")["execution_ms"]
    .agg(["mean", "std", "min", "max"])
)

display(index_summary.round(4))

without_index = index_summary.loc[
    "without_index", "mean"
]

with_index = index_summary.loc[
    "with_index", "mean"
]

speedup = without_index / with_index

index_size = (
    index_results.loc[
        index_results["condition"] == "with_index",
        "index_bytes"
    ].iloc[0] / 1024**2
)

creation_time = index_results.loc[
    index_results["condition"] == "with_index",
    "index_creation_seconds"
].iloc[0]

print(f"Without additional index: {without_index:.3f} ms")
print(f"With additional index: {with_index:.3f} ms")
print(f"Speedup: {speedup:.2f}x")
print(f"Index size: {index_size:.3f} MiB")
print(f"Index creation time: {creation_time:.3f} s")

display(
    index_results[
        ["condition", "scan_type"]
    ].drop_duplicates()
)


,mean,std,min,max
condition,,,,
with_index,0.0733,0.0091,0.063,0.080
without_index,23.6890,0.9247,22.681,24.498


Without additional index: 23.689 ms
With additional index: 0.073 ms
Speedup: 323.03x
Index size: 3.984 MiB
Index creation time: 0.339 s


,condition,scan_type
0,without_index,Seq Scan
3,with_index,Index Scan



The index experiment compares the execution of a selective analytical query before and after creating an additional composite partial index.

The comparison illustrates the trade-off between query execution performance, index creation time, and additional storage requirements.

The experiment verifies that the query returns identical results under both configurations.

The measured performance improvement is specific to the selected query and should not be generalized to all analytical workloads.



## 10. Conclusions

### 10.1 Project Summary

This project developed a reproducible data processing infrastructure for Erasmus+ mobility records covering the period from 2014 to 2023.

The implemented system integrates Python, PostgreSQL, Docker Compose, and GitHub to support structured data storage, automated processing, analytical queries, and computational performance evaluation.

### 10.2 Main Technical Achievements

- Successful processing of 588,109 mobility records.
- Relational database design with integrity constraints and automatically generated attributes.
- Efficient data loading using PostgreSQL COPY.
- SQL-based implementation of representative mobility analyses.
- Validation of aggregate results against the pandas implementation.
- Automated execution through a two-container Docker architecture.
- Performance evaluation across three dataset sizes.

### 10.3 Main Performance Findings

The complete dataset was processed in an average of approximately 3.30 seconds.

Data loading represented the dominant computational stage, while all four analytical SQL queries completed in less than 100 milliseconds on average.

The complete PostgreSQL database occupied approximately 85.58 MiB.

Peak memory consumption was approximately 98.84 MiB for the Python pipeline container and 289.13 MiB for the PostgreSQL container.

The scalability experiments demonstrated approximately linear growth in data loading time and storage requirements across the tested dataset sizes.

### 10.4 Limitations

The experiments were conducted on a single computer using a local Docker environment.

Only three dataset sizes were evaluated.

Memory measurements were performed separately for one full-dataset execution.

The mobility dataset contains aggregated observations, limiting the interpretation of individual mobility behavior.

### 10.5 Final Remarks

The project demonstrates how a conventional exploratory data analysis can be transformed into a structured, automated, and reproducible data science infrastructure.

The combination of relational data storage, containerization, and computational measurements addresses the central objectives of Computational Infrastructures for Data Science.
